<a href="https://colab.research.google.com/github/TongKey-gy/Classifying_Movie_Reviews/blob/main/Base_ex01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights, ResNet50_Weights # ResNet50 가중치 추가
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ==========================================
# 1. 하이퍼파라미터 및 시드 설정
# ==========================================
CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 5,          # 모델이 무거워졌으므로 에폭을 살짝 늘려보는 것도 좋습니다
    'LEARNING_RATE': 1e-3,
    'BATCH_SIZE': 32,     # VRAM이 부족하다면 16으로 줄여주세요
    'SEED': 42
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. 데이터 로드 및 Dataset 정의
# ==========================================
train_df = pd.read_csv('/content/drive/MyDrive/open/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/open/dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = os.path.join(self.root_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views

        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

# [튜닝 포인트 1] 물리 추론 맞춤형 데이터 증강
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5))], p=0.2),
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MultiViewDataset(train_df, '/content/drive/MyDrive/open/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, '/content/drive/MyDrive/open/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

test_df = pd.read_csv('/content/drive/MyDrive/open/sample_submission.csv')
test_dataset = MultiViewDataset(test_df, '/content/drive/MyDrive/open/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# ==========================================
# 3. 모델 정의 (Attention 적용된 ResNet50)
# ==========================================
# [튜닝 포인트 2] 뷰 중요도 판별 Attention 메커니즘 & 깊은 Backbone
class MultiViewAttentionNet(nn.Module):
    def __init__(self, num_classes=1):
        super(MultiViewAttentionNet, self).__init__()
        self.backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])

        feature_dim = 2048

        self.attention = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim // 4),
            nn.ReLU(),
            nn.Linear(feature_dim // 4, 2),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, views):
        f_front = self.feature_extractor(views[0]).view(views[0].size(0), -1)
        f_top = self.feature_extractor(views[1]).view(views[1].size(0), -1)

        combined_features = torch.cat((f_front, f_top), dim=1)
        attn_weights = self.attention(combined_features)

        w_front = attn_weights[:, 0].unsqueeze(1)
        w_top = attn_weights[:, 1].unsqueeze(1)

        f_front_attended = f_front * w_front
        f_top_attended = f_top * w_top

        final_combined = torch.cat((f_front_attended, f_top_attended), dim=1)
        return self.classifier(final_combined)

# ==========================================
# 4. 학습 및 검증 함수
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()

            outputs = model(views).view(-1)
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score

# ==========================================
# 5. 메인 루프 (학습 및 추론)
# ==========================================
# 변경된 모델 적용
model = MultiViewAttentionNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])

print("🚀 학습을 시작합니다...")
for epoch in range(1, CFG['EPOCHS'] + 1):
    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch [{epoch}/{CFG['EPOCHS']}]")
    print(f"  - Train Loss: {avg_train_loss:.4f}")
    print(f"  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}")

# 추론 부분
print("🔍 테스트 데이터 추론을 시작합니다...")
model.eval()
all_probs = []

with torch.no_grad():
    for views in tqdm(test_loader, desc="Inference"):
        views = [v.to(device) for v in views]
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

submission.to_csv('submission.csv', encoding='UTF-8-sig', index=False)
print("🎉 submission.csv 저장 완료!")

학습 데이터 개수: 1000
검증 데이터 개수: 100
🚀 학습을 시작합니다...


Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [1/5]
  - Train Loss: 0.1997
  - Val Log-Loss: 0.669988 | Val Acc: 0.8200


Training:   0%|          | 0/32 [00:00<?, ?it/s]

...

Training:   0%|          | 0/32 [00:00<?, ?it/s]

Validation:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch [5/5]
  - Train Loss: 0.0022
  - Val Log-Loss: 0.586723 | Val Acc: 0.8500
🔍 테스트 데이터 추론을 시작합니다...


Inference:   0%|          | 0/32 [00:00<?, ?it/s]

🎉 submission.csv 저장 완료!
